# Company Data Explorer
By Junyu Gai. This public learning demo uses fictional companies only.

Follow the original data workflow: load and clean, filter and enrich, JSON round-trip, and Flask display. Course materials and real company data are not included.

In [1]:
import json
import pandas as pd
from IPython.display import display
from pipeline import load_companies, OUTPUT_PATH
from app import create_app

# None uses the bundled fictional JSON; set a path to read your own CSV.
INPUT_PATH = None
JSON_PATH = OUTPUT_PATH

## 1. Load and clean
Remove missing names and names that are empty or whitespace-only. Keep duplicate names. Valid names are not trimmed.

In [2]:
df_raw = load_companies(INPUT_PATH)
print('Original row count:', len(df_raw))

df_clean = df_raw.dropna(subset=['name']).copy()
print('Rows removed for null names:', len(df_raw) - len(df_clean))
after_null_removal = len(df_clean)

df_clean = df_clean[df_clean['name'].str.strip() != ''].copy()
print('Rows removed for blank names:', after_null_removal - len(df_clean))
print('Cleaned row count:', len(df_clean))

Original row count: 10
Rows removed for null names: 1
Rows removed for blank names: 2
Cleaned row count: 7


## 2. Filter and add a column
Require both `country_code == 'USA'` and a name starting with `ad` regardless of letter case. `name_length` counts characters, including spaces and punctuation.

In [3]:
only_usa = df_clean['country_code'] == 'USA'
starts_with_ad = df_clean['name'].str.lower().str.startswith('ad', na=False)
df_filtered = df_clean[only_usa & starts_with_ad].copy()

# name_length counts all characters in a company name.
df_filtered['name_length'] = df_filtered['name'].str.len()
print('Filtered row count:', len(df_filtered))
display(df_filtered.head(10))

Filtered row count: 5


,name,country_code,region,city,name_length
0,AdAtlas Example,USA,New York,New York,15
1,adBeacon Example,USA,New York,Albany,16
2,ADCloud Example,USA,California,San Francisco,15
3,AdDelta Example,USA,New York,Buffalo,15
9,AdCafé Example,USA,New York,Rochester,14


## 3. JSON round-trip
`orient='records'` represents each row as an object in a JSON array. Write the records to disk, read the file back, and compare row counts. Matching counts alone do not prove that every value or dtype is unchanged.

In [4]:
records = json.loads(df_filtered.to_json(orient='records'))
JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
with JSON_PATH.open('w', encoding='utf-8') as file:
    json.dump(records, file, indent=2, allow_nan=False)
with JSON_PATH.open(encoding='utf-8') as file:
    reloaded_records = json.load(file)

# Supplying columns keeps the schema even if the filtered list is empty.
df_reloaded = pd.DataFrame.from_records(reloaded_records, columns=df_filtered.columns)
row_counts_match = len(df_filtered) == len(df_reloaded)
print('JSON object count:', len(records))
print(json.dumps(records[:5], indent=2))
# This verifies record-count preservation across the JSON round-trip.
print('Row counts match:', row_counts_match)

JSON object count: 5
[
  {
    "name": "AdAtlas Example",
    "country_code": "USA",
    "region": "New York",
    "city": "New York",
    "name_length": 15
  },
  {
    "name": "adBeacon Example",
    "country_code": "USA",
    "region": "New York",
    "city": "Albany",
    "name_length": 16
  },
  {
    "name": "ADCloud Example",
    "country_code": "USA",
    "region": "California",
    "city": "San Francisco",
    "name_length": 15
  },
  {
    "name": "AdDelta Example",
    "country_code": "USA",
    "region": "New York",
    "city": "Buffalo",
    "name_length": 15
  },
  {
    "name": "AdCaf\u00e9 Example",
    "country_code": "USA",
    "region": "New York",
    "city": "Rochester",
    "name_length": 14
  }
]
Row counts match: True


## 4. Flask webpage
Filter the reloaded table by region, not city. The app returns an HTML table so company details are easy to compare. HTML escaping prevents company text from being interpreted as markup.

In [5]:
df_new_york = df_reloaded[df_reloaded['region'] == 'New York'].copy()
print('New York region row count:', len(df_new_york))
app = create_app(df_new_york)

# Test the Flask route without blocking the notebook with a running server.
response = app.test_client().get('/')
print('Flask response status:', response.status_code)
print('HTML table returned:', '<table' in response.get_data(as_text=True))
display(df_new_york)

New York region row count: 4
Flask response status: 200
HTML table returned: True


,name,country_code,region,city,name_length
0,AdAtlas Example,USA,New York,New York,15
1,adBeacon Example,USA,New York,Albany,16
3,AdDelta Example,USA,New York,Buffalo,15
4,AdCafé Example,USA,New York,Rochester,14


## Visit the live page
From a separate terminal in this project directory, run `python app.py`, then open http://127.0.0.1:5002/. Stop the server with Ctrl+C. The test client above checks the response; it does not start a network server.

Expected fictional-data counts: 10 input, 7 cleaned, 5 filtered, and 4 New York-region records. These are demo counts, not real-company statistics.